In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path.cwd()

OUTPUTS_DIR = PROJECT_DIR / "outputs"

print("=" * 120)
print("SEARCHING FOR H=10 PREDICTION / OOF FILES")
print("=" * 120)

csv_files = list(OUTPUTS_DIR.rglob("*.csv"))

keywords = [
    "h10",
    "oof",
    "prediction",
    "predictions",
    "pooled",
    "fusion",
    "derivative",
    "seed",
    "fold",
]

candidates = []

for path in csv_files:

    path_text = str(path).lower()

    score = sum(
        1 for key in keywords
        if key in path_text
    )

    if score == 0:
        continue

    try:
        df = pd.read_csv(path)
    except Exception:
        continue

    candidates.append(
        {
            "score": score,
            "path": str(path),
            "rows": len(df),
            "columns": list(df.columns),
        }
    )

candidates = sorted(
    candidates,
    key=lambda x: (
        -x["score"],
        x["path"]
    )
)

print(f"\nFound {len(candidates)} candidate CSV files.\n")

for i, item in enumerate(candidates, start=1):

    print("-" * 120)
    print(f"CANDIDATE {i}")
    print(f"Score : {item['score']}")
    print(f"Rows  : {item['rows']}")
    print(f"File  : {item['path']}")
    print("Columns:")

    for col in item["columns"]:
        print(f"  {col}")

print("\n" + "=" * 120)
print("SEARCH COMPLETE")
print("=" * 120)

In [ ]:
# =============================================================================
# FINAL H=10 BINARY FUTURE WELD STATE PREDICTION
# =============================================================================
#
# FINAL CONSISTENT PROTOCOL
#
# Source:
#   Final_Grouped_H10_Ablation_HorizonFolds_5Fold_10Seeds
#   / out_of_fold_predictions.csv
#
# Multiclass model:
#   Fusion-LSTM with derivatives
#
# Binary interpretation:
#
#   Good                 -> Good
#   Burr                 -> Defect
#   Flash-burr           -> Defect
#   Surface-groove/void  -> Defect
#
# IMPORTANT:
#   No separate binary model is trained.
#
#   Binary performance is derived from exactly the same held-out OOF
#   multiclass predictions used for the final H=10 Fusion-LSTM analysis.
#
# Expected:
#   2581 unique H=10 OOF sequences / seed
#   10 seeds (42-51)
#   25810 predictions after configuration filtering
#
# =============================================================================


# =============================================================================
# 1. IMPORTS
# =============================================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
)


# =============================================================================
# 2. PATHS
# =============================================================================

PROJECT_DIR = Path.cwd()

SOURCE_DIR = (
    PROJECT_DIR
    / "outputs"
    / "Final_Grouped_H10_Ablation_HorizonFolds_5Fold_10Seeds"
)

OOF_FILE = (
    SOURCE_DIR
    / "out_of_fold_predictions.csv"
)

OUTPUT_DIR = (
    SOURCE_DIR
    / "Final_H10_Binary_Future_Weld_State"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# =============================================================================
# 3. FINAL EXPERIMENT SETTINGS
# =============================================================================

HORIZON_FRAMES = 10

FPS = 30.0

HORIZON_SECONDS = (
    HORIZON_FRAMES
    / FPS
)

FINAL_CONFIGURATION = (
    "Fusion-LSTM with derivatives"
)

EXPECTED_SEEDS = list(
    range(42, 52)
)

EXPECTED_SEQUENCES_PER_SEED = 2581

EXPECTED_TOTAL_PREDICTIONS = (
    EXPECTED_SEQUENCES_PER_SEED
    * len(EXPECTED_SEEDS)
)

CLASS_ORDER_MULTICLASS = [
    "Good",
    "Burr",
    "Flash-burr",
    "Surface-groove/void",
]

CLASS_ORDER_BINARY = [
    "Good",
    "Defect",
]


# =============================================================================
# 4. BINARY CLASS CONVERSION
# =============================================================================

def to_binary(label):

    label = str(label).strip()

    if label == "Good":
        return "Good"

    if label in {
        "Burr",
        "Burrs",
        "Flash-burr",
        "Flash_burr",
        "Surface-groove/void",
        "Surface_groove_void",
    }:
        return "Defect"

    raise ValueError(
        f"Unexpected multiclass label: {label}"
    )


# =============================================================================
# 5. 95% CONFIDENCE INTERVAL
# =============================================================================

def calculate_mean_sd_ci95(values):

    values = np.asarray(
        values,
        dtype=float,
    )

    mean_value = np.mean(values)

    sd_value = np.std(
        values,
        ddof=1,
    )

    ci_half_width = (
        1.96
        * sd_value
        / np.sqrt(len(values))
    )

    return {
        "mean": mean_value,
        "sd": sd_value,
        "ci95_lower": (
            mean_value
            - ci_half_width
        ),
        "ci95_upper": (
            mean_value
            + ci_half_width
        ),
    }


# =============================================================================
# 6. LOAD FINAL OOF FILE
# =============================================================================

print(
    "\n"
    + "=" * 110
)

print(
    "FINAL H=10 BINARY FUTURE WELD STATE PREDICTION"
)

print(
    "=" * 110
)

print(
    f"\nSource OOF file:\n{OOF_FILE}"
)

if not OOF_FILE.exists():

    raise FileNotFoundError(
        f"\nOOF file not found:\n{OOF_FILE}"
    )

df = pd.read_csv(
    OOF_FILE
)

print(
    f"\nOriginal OOF file shape: {df.shape}"
)


# =============================================================================
# 7. REQUIRED COLUMN AUDIT
# =============================================================================

required_columns = [
    "configuration",
    "number_of_features",
    "seed",
    "fold",
    "sequence_id",
    "exp_id",
    "input_start_frame",
    "input_end_frame",
    "target_frame",
    "true_id",
    "pred_id",
    "true_class",
    "pred_class",
]

missing_columns = [
    col
    for col in required_columns
    if col not in df.columns
]

if missing_columns:

    raise KeyError(
        "\nMissing required columns:\n"
        + "\n".join(missing_columns)
    )

print(
    "\nRequired-column audit PASSED."
)


# =============================================================================
# 8. SHOW CONFIGURATIONS
# =============================================================================

print(
    "\nConfigurations present in source file:"
)

for configuration in sorted(
    df["configuration"]
    .astype(str)
    .unique()
):

    n_rows = (
        df[
            df["configuration"]
            == configuration
        ]
        .shape[0]
    )

    print(
        f"  {configuration}: {n_rows} rows"
    )


# =============================================================================
# 9. FILTER FINAL MODEL
# =============================================================================

final_df = (
    df[
        df["configuration"]
        == FINAL_CONFIGURATION
    ]
    .copy()
)

if final_df.empty:

    raise RuntimeError(
        "\nCould not find exact configuration:\n"
        f"{FINAL_CONFIGURATION}\n\n"
        "Inspect the configuration names printed above."
    )

print(
    "\nSelected final configuration:"
)

print(
    FINAL_CONFIGURATION
)

print(
    f"\nRows after configuration filtering: "
    f"{len(final_df)}"
)


# =============================================================================
# 10. BASIC H=10 AUDIT
# =============================================================================

if len(final_df) != EXPECTED_TOTAL_PREDICTIONS:

    raise RuntimeError(
        "\nFINAL H=10 ROW-COUNT AUDIT FAILED.\n\n"
        f"Expected : {EXPECTED_TOTAL_PREDICTIONS}\n"
        f"Observed : {len(final_df)}"
    )

print(
    "\nTotal-row audit PASSED:"
)

print(
    f"  {EXPECTED_SEQUENCES_PER_SEED} sequences/seed"
    f" × {len(EXPECTED_SEEDS)} seeds"
    f" = {EXPECTED_TOTAL_PREDICTIONS}"
)


# =============================================================================
# 11. SEED AUDIT
# =============================================================================

observed_seeds = sorted(
    final_df["seed"]
    .astype(int)
    .unique()
    .tolist()
)

print(
    "\nObserved seeds:"
)

print(
    observed_seeds
)

if observed_seeds != EXPECTED_SEEDS:

    raise RuntimeError(
        "\nSeed audit FAILED.\n"
        f"Expected: {EXPECTED_SEEDS}\n"
        f"Observed: {observed_seeds}"
    )

print(
    "\nSeed audit PASSED."
)


# =============================================================================
# 12. SEQUENCE COUNT PER SEED
# =============================================================================

seed_counts = (
    final_df
    .groupby("seed")
    .size()
)

print(
    "\nOOF prediction count per seed:"
)

print(
    seed_counts.to_string()
)

if not (
    seed_counts
    == EXPECTED_SEQUENCES_PER_SEED
).all():

    raise RuntimeError(
        "\nSequence-count audit FAILED.\n"
        f"Every seed must contain exactly "
        f"{EXPECTED_SEQUENCES_PER_SEED} OOF sequences."
    )

print(
    "\nSequence-count audit PASSED."
)


# =============================================================================
# 13. FIVE-FOLD AUDIT
# =============================================================================

observed_folds = sorted(
    final_df["fold"]
    .astype(int)
    .unique()
    .tolist()
)

print(
    "\nObserved outer folds:"
)

print(
    observed_folds
)

if len(observed_folds) != 5:

    raise RuntimeError(
        "\nExpected exactly 5 outer folds."
    )

print(
    "\nFive-fold audit PASSED."
)


# =============================================================================
# 14. SAME SEQUENCES ACROSS ALL SEEDS
# =============================================================================

reference_seed = EXPECTED_SEEDS[0]

reference_df = (
    final_df[
        final_df["seed"]
        == reference_seed
    ][
        [
            "sequence_id",
            "exp_id",
            "input_start_frame",
            "input_end_frame",
            "target_frame",
            "true_class",
        ]
    ]
    .sort_values("sequence_id")
    .reset_index(drop=True)
)

for seed in EXPECTED_SEEDS[1:]:

    comparison_df = (
        final_df[
            final_df["seed"]
            == seed
        ][
            [
                "sequence_id",
                "exp_id",
                "input_start_frame",
                "input_end_frame",
                "target_frame",
                "true_class",
            ]
        ]
        .sort_values("sequence_id")
        .reset_index(drop=True)
    )

    if not reference_df.equals(
        comparison_df
    ):

        raise RuntimeError(
            f"\nSequence identity audit FAILED "
            f"for seed {seed}."
        )

print(
    "\nCross-seed sequence identity audit PASSED."
)

print(
    "All 10 seeds evaluate the same H=10 OOF sequences."
)


# =============================================================================
# 15. MULTICLASS LABEL AUDIT
# =============================================================================

true_labels = sorted(
    final_df["true_class"]
    .astype(str)
    .unique()
    .tolist()
)

pred_labels = sorted(
    final_df["pred_class"]
    .astype(str)
    .unique()
    .tolist()
)

print(
    "\nTrue multiclass labels:"
)

print(
    true_labels
)

print(
    "\nPredicted multiclass labels:"
)

print(
    pred_labels
)


# =============================================================================
# 16. CREATE BINARY LABELS
# =============================================================================

final_df[
    "true_binary"
] = (
    final_df["true_class"]
    .apply(to_binary)
)

final_df[
    "pred_binary"
] = (
    final_df["pred_class"]
    .apply(to_binary)
)


# =============================================================================
# 17. SAVE COMPLETE BINARY OOF DATA
# =============================================================================

binary_oof_file = (
    OUTPUT_DIR
    / "H10_binary_all_10_seed_OOF_predictions.csv"
)

final_df.to_csv(
    binary_oof_file,
    index=False,
)


# =============================================================================
# 18. PER-SEED BINARY EVALUATION
# =============================================================================

per_seed_rows = []

confusion_matrices = []

print(
    "\n"
    + "=" * 110
)

print(
    "H=10 BINARY POOLED OOF RESULTS BY SEED"
)

print(
    "=" * 110
)

for seed in EXPECTED_SEEDS:

    seed_df = (
        final_df[
            final_df["seed"]
            == seed
        ]
        .copy()
    )

    y_true = (
        seed_df["true_binary"]
        .to_numpy()
    )

    y_pred = (
        seed_df["pred_binary"]
        .to_numpy()
    )

    accuracy = accuracy_score(
        y_true,
        y_pred,
    )

    (
        macro_precision,
        macro_recall,
        macro_f1,
        _,
    ) = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=CLASS_ORDER_BINARY,
        average="macro",
        zero_division=0,
    )

    (
        weighted_precision,
        weighted_recall,
        weighted_f1,
        _,
    ) = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=CLASS_ORDER_BINARY,
        average="weighted",
        zero_division=0,
    )

    (
        class_precision,
        class_recall,
        class_f1,
        class_support,
    ) = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=CLASS_ORDER_BINARY,
        average=None,
        zero_division=0,
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=CLASS_ORDER_BINARY,
    )

    confusion_matrices.append(
        cm
    )

    row = {
        "seed": seed,

        "accuracy": accuracy,

        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1,

        "weighted_precision": weighted_precision,
        "weighted_recall": weighted_recall,
        "weighted_f1": weighted_f1,

        "good_precision": class_precision[0],
        "good_recall": class_recall[0],
        "good_f1": class_f1[0],
        "good_support": class_support[0],

        "defect_precision": class_precision[1],
        "defect_recall": class_recall[1],
        "defect_f1": class_f1[1],
        "defect_support": class_support[1],
    }

    per_seed_rows.append(
        row
    )

    print(
        f"Seed {seed}: "
        f"Accuracy={accuracy * 100:.2f}% | "
        f"Macro F1={macro_f1 * 100:.2f}% | "
        f"Good Recall={class_recall[0] * 100:.2f}% | "
        f"Defect Recall={class_recall[1] * 100:.2f}%"
    )


per_seed_df = pd.DataFrame(
    per_seed_rows
)

per_seed_df.to_csv(
    OUTPUT_DIR
    / "H10_binary_metrics_by_seed.csv",
    index=False,
)


# =============================================================================
# 19. SUMMARY ACROSS TEN SEEDS
# =============================================================================

summary_metrics = [
    "accuracy",
    "macro_precision",
    "macro_recall",
    "macro_f1",
    "weighted_f1",
    "good_precision",
    "good_recall",
    "good_f1",
    "defect_precision",
    "defect_recall",
    "defect_f1",
]

summary_rows = []

for metric in summary_metrics:

    values = (
        per_seed_df[metric]
        .to_numpy()
        * 100.0
    )

    stats = calculate_mean_sd_ci95(
        values
    )

    summary_rows.append(
        {
            "metric": metric,
            "mean_percent": stats["mean"],
            "sd_percent": stats["sd"],
            "ci95_lower": stats["ci95_lower"],
            "ci95_upper": stats["ci95_upper"],
        }
    )

summary_df = pd.DataFrame(
    summary_rows
)

print(
    "\n"
    + "=" * 110
)

print(
    "FINAL H=10 BINARY POOLED SUMMARY ACROSS 10 SEEDS"
)

print(
    "=" * 110
)

print(
    summary_df.to_string(
        index=False
    )
)

summary_df.to_csv(
    OUTPUT_DIR
    / "H10_binary_summary_10_seeds.csv",
    index=False,
)


# =============================================================================
# 20. AGGREGATED CONFUSION MATRIX
# =============================================================================

aggregated_cm = np.sum(
    confusion_matrices,
    axis=0,
)

print(
    "\n"
    + "=" * 110
)

print(
    "AGGREGATED H=10 BINARY CONFUSION MATRIX ACROSS 10 SEEDS"
)

print(
    "=" * 110
)

print(
    "\nClass order:"
)

print(
    CLASS_ORDER_BINARY
)

print(
    "\nRaw aggregated confusion matrix:"
)

print(
    aggregated_cm
)


# =============================================================================
# 21. NORMALIZED CONFUSION MATRIX
# =============================================================================

normalized_cm = (
    aggregated_cm
    / aggregated_cm.sum(
        axis=1,
        keepdims=True,
    )
    * 100.0
)

print(
    "\nNormalized aggregated confusion matrix (%):"
)

print(
    np.round(
        normalized_cm,
        2,
    )
)

good_recall_aggregated = (
    normalized_cm[0, 0]
)

defect_recall_aggregated = (
    normalized_cm[1, 1]
)

print(
    "\nBinary class-wise recall from aggregated matrix:"
)

print(
    f"Good   : {good_recall_aggregated:.2f}%"
)

print(
    f"Defect : {defect_recall_aggregated:.2f}%"
)


# =============================================================================
# 22. SAVE CONFUSION MATRICES
# =============================================================================

raw_cm_df = pd.DataFrame(
    aggregated_cm,
    index=[
        "Actual Good",
        "Actual Defect",
    ],
    columns=[
        "Predicted Good",
        "Predicted Defect",
    ],
)

raw_cm_df.to_csv(
    OUTPUT_DIR
    / "H10_binary_confusion_matrix_raw.csv"
)


normalized_cm_df = pd.DataFrame(
    normalized_cm,
    index=[
        "Actual Good",
        "Actual Defect",
    ],
    columns=[
        "Predicted Good",
        "Predicted Defect",
    ],
)

normalized_cm_df.to_csv(
    OUTPUT_DIR
    / "H10_binary_confusion_matrix_normalized_percent.csv"
)


# =============================================================================
# 23. PLOT BINARY CONFUSION MATRIX
# =============================================================================

fig, ax = plt.subplots(
    figsize=(6.5, 5.5)
)

im = ax.imshow(
    normalized_cm
)

fig.colorbar(
    im,
    ax=ax,
    label="Percentage (%)",
)

ax.set_xticks(
    [0, 1]
)

ax.set_yticks(
    [0, 1]
)

ax.set_xticklabels(
    CLASS_ORDER_BINARY
)

ax.set_yticklabels(
    CLASS_ORDER_BINARY
)

ax.set_xlabel(
    "Predicted weld state"
)

ax.set_ylabel(
    "Actual weld state"
)

ax.set_title(
    "Future Weld State Prediction at H=10"
)

for i in range(2):

    for j in range(2):

        ax.text(
            j,
            i,
            f"{normalized_cm[i, j]:.1f}%",
            ha="center",
            va="center",
            fontsize=12,
        )

fig.tight_layout()

fig.savefig(
    OUTPUT_DIR
    / "H10_binary_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

plt.close(
    fig
)


# =============================================================================
# 24. MANUSCRIPT CLASS-WISE TABLE
# =============================================================================

manuscript_rows = []

for display_name, prefix in [
    ("Good", "good"),
    ("Defect", "defect"),
]:

    precision_stats = (
        calculate_mean_sd_ci95(
            per_seed_df[
                f"{prefix}_precision"
            ].to_numpy()
            * 100.0
        )
    )

    recall_stats = (
        calculate_mean_sd_ci95(
            per_seed_df[
                f"{prefix}_recall"
            ].to_numpy()
            * 100.0
        )
    )

    f1_stats = (
        calculate_mean_sd_ci95(
            per_seed_df[
                f"{prefix}_f1"
            ].to_numpy()
            * 100.0
        )
    )

    manuscript_rows.append(
        {
            "Future weld state": display_name,

            "Precision (%)": (
                f"{precision_stats['mean']:.2f} "
                f"± {precision_stats['sd']:.2f}"
            ),

            "Recall (%)": (
                f"{recall_stats['mean']:.2f} "
                f"± {recall_stats['sd']:.2f}"
            ),

            "F1-score (%)": (
                f"{f1_stats['mean']:.2f} "
                f"± {f1_stats['sd']:.2f}"
            ),
        }
    )

manuscript_df = pd.DataFrame(
    manuscript_rows
)

print(
    "\n"
    + "=" * 110
)

print(
    "MANUSCRIPT-READY H=10 FUTURE WELD STATE PERFORMANCE"
)

print(
    "=" * 110
)

print(
    manuscript_df.to_string(
        index=False
    )
)

manuscript_df.to_csv(
    OUTPUT_DIR
    / "H10_binary_manuscript_class_performance.csv",
    index=False,
)


# =============================================================================
# 25. EXTRACT MAIN MANUSCRIPT NUMBERS
# =============================================================================

def get_summary_metric(metric_name):

    row = (
        summary_df[
            summary_df["metric"]
            == metric_name
        ]
        .iloc[0]
    )

    return (
        row["mean_percent"],
        row["sd_percent"],
        row["ci95_lower"],
        row["ci95_upper"],
    )


(
    accuracy_mean,
    accuracy_sd,
    accuracy_lower,
    accuracy_upper,
) = get_summary_metric(
    "accuracy"
)

(
    macro_f1_mean,
    macro_f1_sd,
    macro_f1_lower,
    macro_f1_upper,
) = get_summary_metric(
    "macro_f1"
)

(
    defect_recall_mean,
    defect_recall_sd,
    defect_recall_lower,
    defect_recall_upper,
) = get_summary_metric(
    "defect_recall"
)


# =============================================================================
# 26. MANUSCRIPT INTERPRETATION
# =============================================================================

manuscript_text = f"""
At the engineering-selected prediction horizon of H=10 frames
(approximately {HORIZON_SECONDS:.3f} s predictive lead time), future
weld-state performance was evaluated by collapsing Burr, Flash-burr,
and Surface-groove/void predictions into a single Defect state while
retaining Good as the non-defective state. The binary assessment was
derived directly from the pooled out-of-fold predictions of the final
Fusion-LSTM with derivatives; no separate binary classifier was
trained.

Across ten training seeds, the binary Future Weld State Prediction
achieved an accuracy of {accuracy_mean:.2f} ± {accuracy_sd:.2f}% and a
Macro F1 score of {macro_f1_mean:.2f} ± {macro_f1_sd:.2f}%.

From a manufacturing perspective, false Defect predictions may trigger
unnecessary intervention, whereas Defect-to-Good errors are more
critical because they may delay recognition of emerging quality
deterioration. Nevertheless, the Defect recall of
{defect_recall_mean:.2f} ± {defect_recall_sd:.2f}% indicates that most
future defective conditions were identified at the selected forecasting
horizon.

The aggregated ten-seed confusion matrix yielded a Defect recall of
{defect_recall_aggregated:.2f}%.
""".strip()


print(
    "\n"
    + "=" * 110
)

print(
    "MANUSCRIPT INTERPRETATION"
)

print(
    "=" * 110
)

print(
    "\n"
    + manuscript_text
)


with open(
    OUTPUT_DIR
    / "H10_binary_manuscript_text.txt",
    "w",
    encoding="utf-8",
) as f:

    f.write(
        manuscript_text
    )


# =============================================================================
# 27. FINAL AUDIT REPORT
# =============================================================================

audit_text = f"""
FINAL H=10 BINARY FUTURE WELD STATE PREDICTION AUDIT

Source file:
{OOF_FILE}

Configuration:
{FINAL_CONFIGURATION}

Prediction horizon:
H = {HORIZON_FRAMES} frames
Approximate lead time = {HORIZON_SECONDS:.3f} s

Outer experiment folds:
{observed_folds}

Seeds:
{observed_seeds}

Sequences per seed:
{EXPECTED_SEQUENCES_PER_SEED}

Total OOF predictions:
{len(final_df)}

Expected total:
{EXPECTED_TOTAL_PREDICTIONS}

Binary mapping:
Good -> Good
Burr -> Defect
Flash-burr -> Defect
Surface-groove/void -> Defect

Separate binary neural network trained:
No

Binary analysis source:
Final multiclass Fusion-LSTM-with-derivatives OOF predictions.

AUDIT STATUS:
PASSED
""".strip()


with open(
    OUTPUT_DIR
    / "H10_binary_audit.txt",
    "w",
    encoding="utf-8",
) as f:

    f.write(
        audit_text
    )


# =============================================================================
# 28. FINAL OUTPUT
# =============================================================================

print(
    "\n"
    + "=" * 110
)

print(
    "FINAL H=10 BINARY FUTURE WELD STATE ANALYSIS COMPLETE"
)

print(
    "=" * 110
)

print(
    f"\nResults saved to:\n{OUTPUT_DIR}"
)

print(
    "\nCritical protocol checks:"
)

print(
    "  Horizon                    : H=10"
)

print(
    "  Temporal input window      : unchanged from final experiment"
)

print(
    "  Outer folds                : final horizon-selection folds"
)

print(
    "  Final model                : Fusion-LSTM with derivatives"
)

print(
    "  Seeds                      : 42-51"
)

print(
    f"  OOF sequences per seed     : "
    f"{EXPECTED_SEQUENCES_PER_SEED}"
)

print(
    f"  Total filtered predictions : "
    f"{EXPECTED_TOTAL_PREDICTIONS}"
)

print(
    "  Separate binary model      : No"
)

print(
    "\nDone."
)